In [4]:
import pandas as pd
import numpy as np
import math
from itertools import chain
import pickle
import time
import os

#from itables import init_notebook_mode
#init_notebook_mode(all_interactive=True)

## Load previously merged data of laboratory variables __(dm,mb,lb,pc,ms,mr,mic,ce,su,vs)__

In [7]:
def load_merged_data_of_lab_vars():
    #load patient IDs who are considered in this  analysis
    pat_id_df=pd.read_csv('../data/patients_in_analysis.csv.gz',index_col=0)
    # get all pat ids
    all_ids=pat_id_df['USUBJID'].to_list()

    fname='merged_df.csv.gz'
    
    fn=os.path.join('../data/',fname)
    merged_df=pd.read_csv(fn,low_memory=False,index_col=0)

    return merged_df

#data=load_merged_data_of_lab_vars()
data.loc[:,data.columns.str.startswith('cm')].columns

#data=load_merged_data_of_lab_vars()
#data.loc[data['USUBJID']=='TB-1020/1006',['DAY']+data.columns[data.columns.str.startswith('mb_')].tolist()].dropna(how='all',axis=1)


Index([], dtype='object')

##  Add previously created temporal dataframes (__ae,cm,dr_reg__) + static medical history (__mh__) to __merged_df__ (dataframe containing previously merged laboratory variables)


In [2]:
##=========================================  
## For patients, who stopped therapy earlier than the scheduled duration of the study, the cumulative drug doses are set to 0 for those days, 
#. where the drugs weren't taken anymore. This originates from the way the drug regimen was extracted in step s4. 
#  To remedy this problem, forward fill the last cumulative dose for those days.
def ffill_dr_reg_cumul_cols(dr_reg):
    ## Extract dr_reg cumulative columns + DAY and USUBJID
    dr_reg_ffill_cols=['DAY','USUBJID']+dr_reg.columns[dr_reg.columns.str.contains('cumul')].tolist()

    ## 1. Replace the 0s with NaNs==> first therapy day & days where wasn't taken anymore are becoming NaNs
    ## 2. Forward fill ==> only the days without drug threapy get filled with last cumulative dose
    ## 3. Fill NaNs with 0==> fill the first day of therapy with a 0, indicating no drugs were taken yet
    dr_reg_ffill=dr_reg.loc[:,dr_reg_ffill_cols].groupby('USUBJID',as_index=False).apply(lambda x: x.sort_values(by='DAY').replace(0, np.nan).ffill().fillna(0))

    ## Merge the original data with the ffilled drug regimen data
    dr_reg_ffill_=pd.merge(dr_reg.loc[:,~dr_reg.columns.str.contains('cumul')],\
                            dr_reg_ffill.loc[:,dr_reg_ffill_cols],on=['DAY','USUBJID'],how='outer')

    return dr_reg_ffill_

##========================================= 
def concatenate_temporal_data(pats_for_analysis,keep_data_with_unknown_drug_regimen,
                             common_variables_for_analysis,
                             keep_days_with_lab_measurements_only):
    start = time.time()
    ## Load whole dataset
    all_phase_df=load_merged_data_of_lab_vars()
    
    ## Load the last day of therapy of initial treatment
    last_initial_therapy_day_df=pd.read_csv('../data/out_last_initial_therapy_day_list_1018_20_21_22_30.csv.gz',index_col=0)
    
    ## Subset merged dataframe to patients who are considered for analysis
    df_for_anal=all_phase_df.loc[all_phase_df['USUBJID'].isin(pats_for_analysis),:]#.dropna(how='all',axis=1)
    end=time.time()
    t=round((end-start)/60,2)
    print('lab measurement data: done ',t, ' minutes')  
    print('dataframe shape: ',df_for_anal.shape)  

    
    ### LOOP OVER TEMPORAL DATASETS AND ADD THEM TO THE DATAFRAME CREATED IN PREVIOUS STEP

    temp_df_fnames={'dr_reg':'out_temporal_pat_regimens_1018_20_21_22_30.csv.gz',
                    'ae_temp':'out_ae_standardised_temporal.csv.gz',
                    'cm_temp_drugs_doses':'out_cm_temporal_with_doses.csv.gz',
                    'cm_temp_drugs_days_of_appl':'out_cm_temporal_days_of_application.csv.gz',
                    'cm_temp_ind':'out_cm_temporal_indications.csv.gz'}                    

    
    for temp_df_name in [*temp_df_fnames][0:]:

        fn=os.path.join('../data',temp_df_fnames[temp_df_name])
        if temp_df_name=='dr_reg':
            temp_df=pd.read_csv(fn,low_memory=False,index_col=0)    
            
            ## Forward fill cumulative columns
            temp_df=ffill_dr_reg_cumul_cols(temp_df)
            
            #print('dr_reg shape',temp_df.shape)

        ## For these dataframes the pre-selected variables can reduce the numbers of columns that need to be read
        #  Faster read-in time
        if temp_df_name=='ae_temp' or temp_df_name=='cm_temp_ind':
            data_for_cols=pd.read_csv(fn, index_col=0, nrows=0)
            cols_to_read=['DAY','USUBJID']+ list(set(common_variables_for_analysis)&set(data_for_cols.columns))
            temp_df=pd.read_csv(fn,low_memory=False,usecols=cols_to_read,index_col=0)
        
        ## The columns names of these are not in the pre-selected variables, as they are standardised and 
        #  derived in 3_4 -> read all of their columns and drop the unnecessary ones
        if temp_df_name=='cm_temp_drugs_doses' or temp_df_name=='cm_temp_drugs_days_of_appl':  
            temp_df=pd.read_csv(fn,low_memory=False,index_col=0)  


        common_idx=list(set(temp_df['USUBJID'])&set(df_for_anal['USUBJID']))
        temp_df=temp_df.loc[temp_df['USUBJID'].isin(common_idx)].dropna(how='all',axis=1)
        
        if temp_df_name=='dr_reg':
            cols_to_keep=temp_df.columns[temp_df.columns.str.contains('cumul',na=False)].tolist() + ['USUBJID','DAY','STUDYID']
            print('cols_to_keep',cols_to_keep)
            temp_df=temp_df.loc[temp_df['USUBJID'].isin(pats_for_analysis),cols_to_keep].fillna(0)
            dr_reg_max_days=temp_df.loc[:,['USUBJID','DAY']]
            
            ## KEEP ONLY THE DAYS THAT ARE PRESENT IN df_for_anal -> MERGE DATA ONLY FROM THOSE DAYS in dr_reg
            # i.e. Pat 1 has measurements from day 1, 14, 28, 56, 120, 155 in df_for_anal, whereas in dr_reg 
            #      Pat 1 has consecutive data from day 1-day 182 -> 
            #      to merge data from only the days present in df_for_anal, use 'left' as merging method
            if keep_days_with_lab_measurements_only==True:
                merge_method='left'
            
            # MERGE ON THE DAYS THAT ARE PRESENT IN dr_reg -> 
            # i.e. Pat 1 has measurements from day 1, 14, 28, 56, 120, 155 in df_for_anal, whereas in dr_reg 
            #      Pat 1 has consecutive data from day 1-day 182 -> merge an all days from day 1-182 ->
            #      for this use 'outer' as merging method
            if keep_days_with_lab_measurements_only==False:  
                merge_method='outer'

            df_for_anal=pd.merge(df_for_anal,temp_df,left_on=['USUBJID','DAY','STUDYID'],\
                                right_on=['USUBJID','DAY','STUDYID'],how=merge_method)                             

            if keep_data_with_unknown_drug_regimen==False:
                ## KEEP THERAPY RANGE ONLY, WHERE THERE IS RELIABLE DRUG REGIMEN DATA & MICROBIOLOGICAL MEASUREMENTS AVAILABLE
                comm_idx=list(set(dr_reg_max_days['USUBJID'])&set(df_for_anal['USUBJID']))
                days_to_drop=[]
                max_day=0
                for pat in comm_idx[0:]:
                    if pat in last_initial_therapy_day_df['USUBJID'].unique():
                        dr_reg_max=last_initial_therapy_day_df.loc[last_initial_therapy_day_df['USUBJID']==pat,'last_init_therapy_day'].values[0]+10
                    else:
                        #continue
                        dr_reg_max=(dr_reg_max_days[dr_reg_max_days['USUBJID']==pat]['DAY'].max())
                    #mb_max=(df_for_anal[(df_for_anal['USUBJID']==pat)&
                    #                ~(df_for_anal.loc[:,df_for_anal.columns.str.startswith('mb_')].isna().all(axis=1))]['DAY'].max())
                    max_day_to_keep=dr_reg_max #max(mb_max,dr_reg_max) # 
                    days_to_drop.append(df_for_anal[(df_for_anal['USUBJID']==pat)&(df_for_anal['DAY']>max_day_to_keep)].index.tolist())
                    if max_day_to_keep>max_day:
                        max_day=max_day_to_keep
                days_to_drop=list(chain(*days_to_drop))
                df_for_anal=df_for_anal.drop(index=days_to_drop)

            print(temp_df_name,"df_for_anal['DAY'].max()",df_for_anal['DAY'].max())
            

        if temp_df_name!='dr_reg':                
            df_for_anal=pd.merge(df_for_anal,temp_df,left_on=['USUBJID','DAY','STUDYID'],\
                                    right_on=['USUBJID','DAY','STUDYID'],how='left')

            print(temp_df_name,"df_for_anal['DAY'].max()",df_for_anal['DAY'].max())
        
        del temp_df
        end=time.time()
        t=round((end-start)/60,2)
        print(temp_df_name+' done ',t, ' minutes')  
        print('dataframe shape: ',df_for_anal.shape) 
             

    print('max_day',df_for_anal['DAY'].max())

    ###--------------------
    #### LOAD EXPANDED MEDICAL HISTORY DATAFRAME AND ADD COLUMNS FOR EACH PATIENT 
    mh=pd.read_csv('../data/out_mh_standardised_expanded.csv.gz',low_memory=False,index_col=0)
    ## Drop terms with less than 10 patients having that term in medical history
    mh=mh.loc[:,~mh.columns.str.contains('STUDYID')]
    #mh=mh.loc[:,mh.sum()>10]
    
    common_idx=list(set(mh.index)&set(df_for_anal['USUBJID']))
    mh=mh.loc[common_idx,:].dropna(how='all',axis=1)

    mh_colnames_with_prefix=['mh_'+x for x in mh.columns.tolist()]
    mh.columns=mh_colnames_with_prefix
    common_cols=list(set(mh.columns)&set(common_variables_for_analysis))
    mh=mh.loc[:,common_cols]
    #mh=mh.rename_axis('USUBJID').reset_index()

    ## Add common columns with Nans
    df_for_anal[common_cols]=np.nan
    
    ## Select rows of patients in df_for_anal with MH data
    ids_with_mh=df_for_anal.loc[df_for_anal['USUBJID'].isin(common_idx),'USUBJID'].index.tolist()
    df_for_anal.loc[ids_with_mh,common_cols]=mh.loc[df_for_anal.loc[ids_with_mh,'USUBJID'].tolist(),common_cols].values
    del mh        
    #df_for_anal=df_for_anal.drop(columns=['mh_STUDYID'])

    end=time.time()
    t=round((end-start)/60,2)
    print('mh done ',t, ' minutes')
    print('dataframe shape: ',df_for_anal.shape) 
    
    
    ### Fill NaN datapoints with 0 where no imputation is needed 
    #  (cmdos and cmday columns need forward fill as they are cumulative doses/days of drug application)
    #df_for_anal=df_for_anal.sparse.to_dense()
    df_for_anal.loc[:,df_for_anal.columns.str.startswith(('ae_','cmind_','mh_'))]=df_for_anal.loc[:,df_for_anal.columns.str.startswith(('ae_','cmind_','mh_'))].fillna(0)

    end=time.time()
    t=round((end-start)/60,2)
    print('fillna with 0s done ',t, ' minutes') 
       
    return df_for_anal


## Select columns for analysis

In [3]:
def select_columns_for_analysis(data,pats_for_analysis,common_variables_for_analysis,return_selected_columns_only):
    # Read lab test variable type information (numerical or categorical)
    with open('../data/lab_variables.pkl', 'rb') as f:
        lab_variable_type_dict = pickle.load(f)
    
    ## Correct typos and drop some duplicated columns
    #data.columns = ['WEEK' if x=='Week' else x for x in data.columns]
    
    ### SELECT COLUMNS THAT HOLD THE RESULT INFROMATION USED IN THE ANALYSIS FOR EACH DATASET TYPE
    dataset_types_of_common_vars=list(set([x.split('_')[0] for x in common_variables_for_analysis if '_' in x]))

    result_column_names_per_dataset_type={  
                
                'mb':{'categorical':{'categorical_vars':[# Raw variable names of all mb tests
                                                        'Identification','Culture Growth','Categorical Count','Unknown',\
                                                        'Colony Count, Categorical','MPT64 Antigen Test',
                                                         # Standardised variable names 
                                                         'ZN-smear','MGIT','HAIN-test','MTB-complex',
                                                         'Auramine-smear','LJ-culture','AccuProbe',
                                                         'MPT64-Antigen-Test','RT-PCR'],
                                     
                                    'categorical_result_suffix':['STD_RESULT','CULTURE_STATUS','STD_CAT_ORDINAL_RESULT']}, #'RESULT if using all mb variables, STD_RESULT if using the standardised ones
                    'numerical':{'numerical_vars':['Time to Detection','Colony Count','Minimum Cycle Threshold of Detection','MGIT','LJ-culture'],
                                'numerical_result_suffix':['STD_NUM_RESULT']}},

                'pc':{'numerical':{'numerical_result_suffix':['STD_NUM_RESULT']}},

                're':{'categorical':{'categorical_vars':['Cavitation','X-Ray compatible with TB','Development of New Lesions',\
                                                        'Extension of Old Lesions','Extent of Disease','Interpretation',\
                                                        'Size Of Cavitation'],
                                    'categorical_result_suffix':['STD_CAT_ORDINAL_RESULT']},
                    'numerical':{'numerical_vars':['Detailed Classification'],
                                'numerical_result_suffix':['STD_NUM_RESULT']}},
        
                'vs':{'numerical':{'numerical_result_suffix':['STD_NUM_RESULT']}},

                'lb':{'categorical':{'categorical_vars':lab_variable_type_dict['lb_categorical_vars'],
                                    'categorical_result_suffix':['STD_CAT_ORDINAL_RESULT']},
                    'numerical':{'numerical_vars':lab_variable_type_dict['lb_numerical_vars'],
                                'numerical_result_suffix':['STD_NUM_RESULT']}},

                #'mh':{'categorical':{'categorical_result_suffix':}},

                'ce':{'categorical':{'categorical_result_suffix':['STD_CAT_ORDINAL_RESULT']}},

                'mr':{'categorical':{'categorical_result_suffix':['STD_CAT_ORDINAL_RESULT']}},

                'mic':{'numerical':{'numerical_result_suffix':['STD_NUM_RESULT']}},

                'ms':{'categorical':{'categorical_result_suffix':['STD_CAT_ORDINAL_RESULT']},
                    'numerical':{'numerical_result_suffix':['SUSC_CONC','RESISTANCE_CONC']}},

                'su':{'categorical':{'categorical_result_suffix':['STD_CAT_ORDINAL_RESULT']}}}

    ### Collect the columns to keep for analysis from the dataframe containing the previously selected variables
    columns_for_analysis={}

    ## Loop over the common variables among patients selected in previous step, and using the dictionary above
    #  extract the columns that hold usable result information for the analysis
    datasets_to_loop_over=set([*result_column_names_per_dataset_type])&set(dataset_types_of_common_vars)

    for dataset_type in datasets_to_loop_over:
        columns_for_analysis[dataset_type]=[]
        dataset_type_vars_all=[x for x in common_variables_for_analysis if x.startswith(dataset_type+'_')]
        
        for dataset_type_var in dataset_type_vars_all:
            stripped_dataset_type_var=dataset_type_var.split('_')[1]

            for var_type in result_column_names_per_dataset_type[dataset_type].keys():

                if var_type+'_vars' in result_column_names_per_dataset_type[dataset_type][var_type].keys():

                    if stripped_dataset_type_var in result_column_names_per_dataset_type[dataset_type][var_type][var_type+'_vars']:
                        for suffix in result_column_names_per_dataset_type[dataset_type][var_type][var_type+'_result_suffix']:
                            columns_for_analysis[dataset_type].append('_'.join([dataset_type_var,suffix]))

                elif var_type+'_vars' not in result_column_names_per_dataset_type[dataset_type][var_type].keys(): 
                    for suffix in result_column_names_per_dataset_type[dataset_type][var_type][var_type+'_result_suffix']:
                        columns_for_analysis[dataset_type].append('_'.join([dataset_type_var,suffix]))


    ## Extract column holding drug regimen cumulative dose information
    temporal_cols=data.columns[data.columns.str.startswith(('ae_','cmdos_','cmday_','cmind_','mh_','dr_reg_'))].tolist()

    ## Create one list of the columns_for_analysis dictionary values containing variables from the different dataset types
    columns_for_analysis_list=list(chain(*np.array(list(columns_for_analysis.values()),dtype=object)))

    ## Columns names of patients extracted from dm dataframe
    dm_colnames=['USUBJID','DAY','STUDYID','ARM','AGE','SEX']
    cols_for_analysis=dm_colnames+ columns_for_analysis_list + temporal_cols


    ## If return_columns_only -> select and return COLUMNS ONLY for analysis from the provided dataset
    if return_selected_columns_only==True:

        data_for_anal=data.loc[:,data.columns.isin(cols_for_analysis)]

        ## Drop mb columns, that are culture based and don't give an instant result,
        #  or the method is unknown -> KEEP MGIT MEASUREMENTS, AS THEY ARE THE GOAL OF PREDICTION
        #mb_cols_to_drop='|'.join(['mb_Culture Growth','mb_Categorical Count','mb_Colony Count','mb_Colony Count, Categorical','mb_Unknown'])
        cols_for_analysis=data_for_anal.columns
                
        return cols_for_analysis

    
    ## If return_columns_only is False -> select columns for analysis and return the DATASET with selected columns
    if return_selected_columns_only==False:
        # Drop all NaN columns and duplicates
        data_for_anal=data.loc[data['USUBJID'].isin(pats_for_analysis),cols_for_analysis].dropna(how='all',axis=1)

        ## Drop mb columns, that are culture based and don't give an instant result,
        #  or the method is unknown -> KEEP MGIT MEASUREMENTS, AS THEY ARE THE GOAL OF PREDICTION
        #mb_cols_to_drop='|'.join(['mb_Culture Growth','mb_Categorical Count','mb_Colony Count','mb_Colony Count, Categorical','mb_Unknown'])
        #data_for_anal=data_for_anal.loc[:,~data_for_anal.columns.str.contains(mb_cols_to_drop,na=False)]
        
        
        return data_for_anal


## Imputation of missing entries 
* #### Forward and backward fill of categorical variables
* #### Linear interpolation of numerical variables 

In [4]:
def imputation(data_for_anal,nan_thr_ratio,only_timepoints_with_mgit):
    import warnings
    #from pandas.core.common import SettingWithCopyWarning
    warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
    warnings.simplefilter(action="ignore", category=pd.errors.SettingWithCopyWarning)
    warnings.filterwarnings("ignore")

    print('Starting imputation')
    ## Convert numbers, that are stored as strings to numeric datatype
    data_for_anal=data_for_anal.apply(lambda x: pd.to_numeric(x,errors='ignore'))


    ## Get columns that have NaN values in them for imputation
    data_for_anal = data_for_anal.loc[:,~data_for_anal.columns.duplicated()]
    

    ## Create dataframe for imputed data
    data_for_anal_imp=data_for_anal.reset_index()

    ### ============ FORWARD FILL MB VARIABLES ============
    ## Forward fill last observed value of the preprocessed & standardised mb test results (mb_test_name_STD_RESULT)
    #. In TB-1022, mb measurements were not undertakedn at every visit, therefore there are NaN datapoints for these mb vars.
    #  As an imputation method, forward fill with last observed value
    mb_colnames=data_for_anal_imp.columns[data_for_anal_imp.columns.str.startswith('mb_')&(data_for_anal_imp.columns.str.endswith('_STD_RESULT|CULTURE_STATUS|STD_CAT_ORDINAL_RESULT'))].tolist()
    
    mb_ffilled=data_for_anal_imp.groupby('USUBJID').apply(lambda x:x.sort_values(by=['DAY']).loc[:,['DAY']+mb_colnames].ffill())
    mb_ffilled=mb_ffilled.reset_index().drop(columns='level_1')
    
    data_for_anal_imp=pd.merge(data_for_anal_imp.loc[:,[col for col in data_for_anal_imp.columns if col not in mb_colnames]],\
                               mb_ffilled.loc[:,['DAY','USUBJID']+mb_colnames],on=['DAY','USUBJID'],how='outer')


    ### ============ FORWARD FILL DR_REG CUMULATIVE VARIABLES ============
    ## Forward fill last observed value of the cumulative drug regimen data 
    ## The dataframe was created in a way, that for study days, where the drug was not taken anymore, the cumulative dose is NaN
    #. ==> ffill the cumulative dose, because that is the real value and not NaN
    dr_cumul_colnames=data_for_anal_imp.columns[data_for_anal_imp.columns.str.startswith('dr_reg')&\
                                                (data_for_anal_imp.columns.str.contains('cumul'))].tolist()
    
    dr_cumul_ffilled=data_for_anal_imp.groupby('USUBJID').apply(lambda x:x.sort_values(by=['DAY']).loc[:,['DAY']+dr_cumul_colnames].ffill())
    dr_cumul_ffilled=dr_cumul_ffilled.reset_index().drop(columns='level_1')
    
    data_for_anal_imp=pd.merge(data_for_anal_imp.loc[:,[col for col in data_for_anal_imp.columns if col not in dr_cumul_colnames]],\
                               dr_cumul_ffilled.loc[:,['DAY','USUBJID']+dr_cumul_colnames],on=['DAY','USUBJID'],how='outer')
    
 


    ### ============ INTERPOLATE NUMERICAL LAB VARIABLES ============
    ## Initialise column names that are numerical and makes sense to interpolate the missing values
    cols_for_num_interpolation='|'.join(['STD_NUM_RESULT_SCALED','STD_NUM_RESULT'])

    ## Extract patients to loop over
    pats=list(set(data_for_anal_imp['USUBJID']))

    ## Create columns for cumulative cm results
    cm_drug_cols=data_for_anal_imp.columns[data_for_anal_imp.columns.str.contains('cmday|cmdos')]
    cm_drug_cumul_cols=[x+'_cumul' for x in cm_drug_cols]
    data_for_anal_imp.loc[:,cm_drug_cumul_cols]=np.nan

    print('data_for_anal_imp.shape before imputation',data_for_anal_imp.shape)

    
    for pat_id in pats[:]:
        pat_df=data_for_anal_imp.loc[data_for_anal_imp['USUBJID']==pat_id,:]

        ## Sort by day
        pat_df.loc[:,'DAY']=pat_df.loc[:,'DAY'].astype(np.float64)
        pat_df=pat_df.sort_values(by=['DAY'])
        
        ### 1. FILL DR_REG COLUMNS WITH 0 IF DRUG WAS NOT APPLIED ON STUDY DAY (DAYS  BEFORE DRUG THERAPY STARTED)
        ### 2. SELECT AND DROP TIMEPOINTS WHERE NO RELIABLE DRUG REGIMEN DATA IS AVAILABLE 
        ## For each dr_reg column, fill the days where no drug was taken (before star_day) with zero
        dr_reg_cols=pat_df.columns[pat_df.columns.str.contains('dr_reg')]
        #print(pat_df[dr_reg_cols])
        for dr_reg_col in dr_reg_cols:
            start_day=pat_df.loc[~pat_df[dr_reg_col].isna(),'DAY'].min()
            pat_df.loc[pat_df['DAY']<start_day,dr_reg_col]=0

        
        
        ## Select rows (days), where dr_reg data for all drugs is NaN -> no reliable dr_reg data available for those days! 
        rows_to_drop=pat_df.index[pat_df[dr_reg_cols].isna().all(axis=1).values]
        #print(pat_df[dr_reg_cols])
        #print('len(rows_to_drop)',len(rows_to_drop))
        #pat_df=pat_df.drop(index=rows_to_drop)
        #data_for_anal_imp=data_for_anal_imp.drop(index=rows_to_drop)


        ### 1. FILL NANS WITH 0s IN THE CM COLUMNS THAT CONTAIN DAILY DOSAGE (cmdos)/OR DAILY APPLICATION(cmday) ->
        #      IF NAN, DRUG WAS NOT TAKEN THAT DAY
        cm_drug_cols=pat_df.columns[pat_df.columns.str.contains('cmday|cmdos',na=False)&\
                                    ~pat_df.columns.str.contains('cumul',na=False)]
        pat_df.loc[:,cm_drug_cols]=pat_df.loc[:,cm_drug_cols].fillna(0)
        
        ###  2. CREATE COLUMN WITH CUMULATIVE CM-DRUG INFORMATION
        cm_drug_cols_cumul=[x+'_cumul' for x in cm_drug_cols]
        pat_df.loc[:,cm_drug_cols_cumul]=pat_df.loc[:,cm_drug_cols].cumsum().values
        #print(pat_id,(pat_df.shape))

        ## UPDATE data_for_anal_imp WITH NEWLY CREATED CMDAY|CMDOS|DR_REG INFORMATION
        data_for_anal_imp.loc[pat_df.index,pat_df.columns.str.contains('cmday|cmdos|dr_reg',na=False)]=pat_df.loc[:,pat_df.columns.str.contains('cmday|cmdos|dr_reg',na=False)].values

        ## SELECT COLUMNS WITH NANS FOR IMPUTATION
        cols_for_imputation=pat_df.columns[pat_df.isnull().any().values].tolist() +['DAY']
        #cols_for_imputation=list(set([x for x in cols_for_imputation if x!='mb_Time to Detection_STD_NUM_RESULT']))
        pat_df=data_for_anal_imp.loc[data_for_anal_imp['USUBJID']==pat_id,cols_for_imputation]
        pat_df.loc[:,'DAY']=pat_df.loc[:,'DAY'].astype(np.float64)
        pat_df=pat_df.sort_values(by=['DAY'])

        ### SELECT NUMERICAL COLUMNS WHERE LINEAR INTERPOLATION MAKES SENSE (LAB VALUES MOSTLY) AND RUN INTERPOLATION
        ## Split into numerical and string variables and interpolate numerical variables
        pat_numerical=pat_df.loc[:,pat_df.columns.str.contains(cols_for_num_interpolation,na=False)]

        # Leave out the MGIT results (=='mb_Time to Detection_STD_NUM_RESULT') from imputation
        columns_to_interpolate=[x for x in pat_numerical.columns] #if x!='mb_Time to Detection_STD_NUM_RESULT']
        data_for_anal_imp.loc[pat_numerical.index,columns_to_interpolate]=data_for_anal_imp.loc[pat_numerical.index,columns_to_interpolate].interpolate(method='linear', limit_direction='forward', axis=0)

        ## Drop those days, where the interpolation of numerical lab values was not possible  
        ## ==> initial period in study with no measured values, first measured values in a later timepoint
        #std_num_res_cols=pat_df.columns[pat_df.columns.str.contains(cols_for_num_interpolation)].tolist()
        #data_for_anal_imp = data_for_anal_imp.dropna(subset=std_num_res_cols,how='any')

        ### FORWARD AND BACKWARD FILL ALL THE COLUMNS THAT HAVE NANS
        pat_df=data_for_anal_imp.loc[data_for_anal_imp['USUBJID']==pat_id,cols_for_imputation]
        cols_for_imputation=pat_df.columns[pat_df.isnull().any().values].tolist() +['DAY']
        pat_df_for_bfill_ffill=data_for_anal_imp.loc[data_for_anal_imp['USUBJID']==pat_id,cols_for_imputation].sort_values(by='DAY',ascending=True)

        #Save original index
        pat_df_for_bfill_ffill['original_index']=pat_df_for_bfill_ffill.index.values

        # Set DAY as numerical index and forward + backward fill all variables
        pat_day_index=pat_df_for_bfill_ffill.set_index('DAY',drop=False).sort_index()
        ffill_df=pat_day_index.interpolate('ffill')
        bfill_df=ffill_df.copy() #ffill_df.interpolate('bfill')
        bfill_df=bfill_df.set_index('original_index',drop=True)
        data_for_anal_imp.loc[pat_df_for_bfill_ffill.index,pat_df_for_bfill_ffill.columns]=bfill_df

    #print(data_for_anal_imp)
    ## Drop cmdos|cmday columns that are not cumulative data as the cumulative columns are more important for analysis
    cm_drug_cols_to_drop=data_for_anal_imp.columns[data_for_anal_imp.columns.str.contains('cmday|cmdos',na=False)&\
                                                  ~data_for_anal_imp.columns.str.contains('cumul',na=False)]
    data_for_anal_imp=data_for_anal_imp.drop(columns=cm_drug_cols_to_drop)

    ## If only_timepoints_with_mgit==True, consider timepoints only where an MGIT measurement is available
    if only_timepoints_with_mgit==True:
        data_for_anal_imp=data_for_anal_imp.loc[~(data_for_anal_imp['mb_Time to Detection_STD_NUM_RESULT'].isna()),:]
    
    ## Drop duplicates and drop unnecessary temporary columns
    data_for_anal_imp=data_for_anal_imp.drop_duplicates()
    print('data_for_anal_imp.shape after interpolation & droppping duplicates',data_for_anal_imp.shape)
    data_for_anal_imp=data_for_anal_imp.drop(columns=['original_index'])
    data_for_anal_imp=data_for_anal_imp.dropna(how='all',axis=1)
    print("data_for_anal_imp.shape after data_for_anal_imp.dropna(how='all',axis=1)",data_for_anal_imp.shape)

    ## Drop columns which contain  a significant amount of NaNs
    nan_threshold=nan_thr_ratio*len(data_for_anal_imp['USUBJID'].unique())
    a=data_for_anal_imp.groupby('USUBJID').apply(lambda x: x.isnull().all())
    cols_with_nans_over_thr=a.sum()[a.sum()>nan_threshold].index.tolist()
    data_for_anal_imp=data_for_anal_imp.drop(columns=cols_with_nans_over_thr)
    print(f'Dropped following columns containing only NaNs for more than {100*nan_thr_ratio}% of the patients: {cols_with_nans_over_thr}')
    print("data_for_anal_imp.shape after dropping colsuwms with over the threshold Nans",data_for_anal_imp.shape)
    
    return data_for_anal_imp


                   


'''
selection_method='phase'

## Select the common variables and common patients
if selection_method=='phase':
    num_of_common_vars=59
    phase='1'
    with open('../data/common_vars.pickle', 'rb') as f:
        common_vars=pickle.load(f)
    common_variables_for_analysis=common_vars[phase][num_of_common_vars]['common_variables']+['USUBJID','DAY']
    pats_for_analysis=common_vars[phase][num_of_common_vars]['patients']

filename='data_lab_meas_days_only_with_selected_cols.csv.gz'
data,rows_with_nans=preprocess_data(filename,pats_for_analysis,common_variables_for_analysis,\
                     only_timepoints_with_mgit=True,return_selected_columns_only=True)

'TB-1021/2855311'
'''
                           

"\nselection_method='phase'\n\n## Select the common variables and common patients\nif selection_method=='phase':\n    num_of_common_vars=59\n    phase='1'\n    with open('../data/common_vars.pickle', 'rb') as f:\n        common_vars=pickle.load(f)\n    common_variables_for_analysis=common_vars[phase][num_of_common_vars]['common_variables']+['USUBJID','DAY']\n    pats_for_analysis=common_vars[phase][num_of_common_vars]['patients']\n\nfilename='data_lab_meas_days_only_with_selected_cols.csv.gz'\ndata,rows_with_nans=preprocess_data(filename,pats_for_analysis,common_variables_for_analysis,                     only_timepoints_with_mgit=True,return_selected_columns_only=True)\n\n'TB-1021/2855311'\n"

## Check if there are any NaNs after imputation and drop those patients

In [5]:
def drop_nans(data_for_anal_imp):
    nan_cols=data_for_anal_imp.loc[:,(data_for_anal_imp.isna().any())].columns.tolist()
    pats_with_nan_df=data_for_anal_imp.loc[data_for_anal_imp.loc[:,nan_cols].isna().any(axis=1),['DAY','USUBJID']+nan_cols]
    rows_with_nan=pats_with_nan_df.index.tolist()

    data_for_anal_imp=data_for_anal_imp[~data_for_anal_imp.index.isin(rows_with_nan)]
    print('Shape of data with NaNs :',pats_with_nan_df.shape)
    print('Shape of imputed dataframe :',data_for_anal_imp.shape)

    return data_for_anal_imp,pats_with_nan_df

## Create AnnData from merged dataset containing all data

In [6]:
def create_ann_data(fname,fname_for_saving):
    import ehrapy as ep
    all_phase_df_for_anal=pd.read_csv(fname,low_memory=False,index_col=0)
    all_phase_df_for_anal=all_phase_df_for_anal.reset_index()

    adata = ep.ad.df_to_anndata(
        all_phase_df_for_anal, index_column="index", columns_obs_only=['DAY','ARM','STUDYID','USUBJID'])

    del all_phase_df_for_anal
    ep.io.write('../data/'+fname_for_saving+'.h5ad', adata)   

'''
fname='../data/data_lab_meas_days_only_with_selected_cols.csv.gz'
fname_for_saving='data_lab_meas_days_only_with_selected_cols'
create_ann_data(fname,fname_for_saving)
'''

"\nfname='../data/data_lab_meas_days_only_with_selected_cols.csv.gz'\nfname_for_saving='data_lab_meas_days_only_with_selected_cols'\ncreate_ann_data(fname,fname_for_saving)\n"

## Merge all data into one dataframe + select only columns considered for analysis

* #### __data_lab_meas_days_only_with_selected_cols.csv__: keep only days where lab measurements were made

* #### __data_all_therapy_days_with_selected_cols.csv__: keep all days where drug was applied


In [14]:
def merge_all_data_from_phases_with_cols_for_analysis(keep_days_with_lab_measurements_only,fname,\
                                                      keep_data_with_unknown_drug_regimen,return_selected_columns_only):  

    ## Load and merge lab measurement data
    all_phase_df=load_merged_data_of_lab_vars()
    pats_for_analysis=list(set(all_phase_df['USUBJID']))
    #pats_for_analysis=pats_for_analysis[0:50]
    del all_phase_df


    ## Create list of available variables considered in analysis
    variables_per_patient_all=pd.read_csv('../data/all_pat_variables.csv.gz',index_col=0,low_memory=False)
    vars_per_pat_ttp=variables_per_patient_all.loc[pats_for_analysis,:].dropna(how='all',axis=1)

    vars_available_in_patients=vars_per_pat_ttp.sum(axis=0).sort_values(ascending=False)
    vars_available_in_patients=vars_available_in_patients[vars_available_in_patients >=2] ###:change number to reduce var numbers
    common_variables_for_analysis=vars_available_in_patients.index.tolist()
    print('len(common_variables_for_analysis)',len(common_variables_for_analysis))
    del vars_per_pat_ttp

    ## Merge temporal data with data so far
    all_phase_df_with_temporal=concatenate_temporal_data(pats_for_analysis,keep_data_with_unknown_drug_regimen,\
                                                         common_variables_for_analysis,\
                                                         keep_days_with_lab_measurements_only)
    
    print('Temporal data merged')
    
    ## Select columns for analysis
    data_for_anal=select_columns_for_analysis(all_phase_df_with_temporal,pats_for_analysis,common_variables_for_analysis,\
                                              return_selected_columns_only)
    print('Columns selected')
    print(data_for_anal.shape)
    #del all_phase_df_with_temporal

    data_for_anal.to_csv('../data/'+fname+'.csv.gz',compression='gzip')
    print('Max day',data_for_anal['DAY'].max())
    #del data_for_anal
    
    return data_for_anal
'''

for keep_days_with_lab_measurements_only,fname in zip([True,False][0:1],\
                                                    ['data_lab_meas_days_only_with_selected_cols',\
                                                     'data_all_therapy_days_with_selected_cols']):

    
    data_for_anal=merge_all_data_from_phases_with_cols_for_analysis(keep_days_with_lab_measurements_only,fname,\
                                                      keep_data_with_unknown_drug_regimen=False,\
                                                      return_selected_columns_only=False)

    #print(all_phase_df_with_temporal.shape)
'''    
                                                                                                  

"\n\nfor keep_days_with_lab_measurements_only,fname in zip([True,False][0:1],                                                    ['data_lab_meas_days_only_with_selected_cols',                                                     'data_all_therapy_days_with_selected_cols']):\n\n    \n    data_for_anal=merge_all_data_from_phases_with_cols_for_analysis(keep_days_with_lab_measurements_only,fname,                                                      keep_data_with_unknown_drug_regimen=False,                                                      return_selected_columns_only=False)\n\n    #print(all_phase_df_with_temporal.shape)\n"

## From the dataframe containig all data merged previously, load only the rows and columns with patients and variables that are considered in analysis

In [12]:
def load_selected_patients_and_vars_for_analysis(filename,pats_for_analysis,common_variables_for_analysis,
                                                 return_selected_columns_only):
    fname=os.path.join('../data/',filename)

    ## Select rows to load
    data_index=pd.read_csv(fname,usecols=[1],index_col=False,low_memory=False)
    rows_to_load=[0] + [x+1 for x in data_index[data_index['USUBJID'].isin(pats_for_analysis)].index.tolist()]

    ## Select columns to load
    data_for_cols=pd.read_csv(fname, index_col=0, nrows=0)
    cols_to_load=select_columns_for_analysis(data_for_cols,pats_for_analysis,common_variables_for_analysis,\
                                              return_selected_columns_only)
    
    ## Load data with selected rows and columns
    data=pd.read_csv(fname, skiprows=lambda x: x not in rows_to_load, usecols=cols_to_load.tolist(), 
                     low_memory=False,index_col=False)
                         
    return data

## Bring all functions together under one function to create the preprocessed data for the patients considered in analysis 
- __use this function of ML method can't deal with NaNs__, as there is interpolation of lab variables + columns with NaNs in more than threshold% of patients are dropped

In [13]:
def preprocess_data_with_imputation(filename,pats_for_analysis,common_variables_for_analysis,nan_thr_ratio,\
                                    only_timepoints_with_mgit,return_selected_columns_only):
    
    ## Subset whole data to patients consideres in analysis
    data=load_selected_patients_and_vars_for_analysis(filename,pats_for_analysis,\
                                                     common_variables_for_analysis,return_selected_columns_only)  
    #return data 
    print('mb columns:',data.columns[data.columns.str.startswith('mb')])
    print('data.shape after loading:',data.shape)

    print(data[['DAY']].max())
    ## Drop temporal columns with only 0s and NaNs -> that means column doesn't hold information
    data_temp_cols=data.loc[:,data.columns.str.startswith(('cmind','cmdos','cmday','ae','mh'))]
    temp_cols_to_drop=data_temp_cols.columns[~data_temp_cols.apply(lambda x: x.sum()>0).values]
    data=data.drop(columns=temp_cols_to_drop)

    print('mb columns:',data.columns[data.columns.str.startswith('mb')])
    print('data.shape after dropping temporal cols:',data.shape)

    ## Drop rows with no DAY information 
    data=data[~data['DAY'].isna()]
    print('Data subsetted to patients considered for analysis')   
    print('mb columns:',data.columns[data.columns.str.startswith('mb')])
    print('data.shape after dropping rows with no DAY info:',data.shape)

    ## Replace characters ('[]>') not allowed in column names for XGBoost model
    data.columns=data.columns.str.replace(r'\[REDACTED\]','',regex=False)

    ## Keep only columns with cumulative cmd data (from cmday or cmdos data)
    cm_cols_to_drop=data.columns[data.columns.str.startswith(('cmdos','cmday'))&(data.columns.str.contains('cumul',na=False))]
    data=data.drop(columns=cm_cols_to_drop)

    #print('ce columns:',data.columns[data.columns.str.startswith('ce')])
    print('data.shape after keeping only cumul temporal cols :',data.shape)
    
    ## Convert mb result columns to ordinal result from positive/negative
    '''
    mb_cols_to_ordinal=['mb_Identification_RESULT',',mb_MPT64 Antigen Test_RESULT',
                        'mb_Minimum Cycle Threshold of Detection_RESULT',
                        'mb_Culture Growth_RESULT',
                        'mb_Colony Count, Categorical_RESULT',
                        'mb_Categorical Count_RESULT',
                        'mb_Unknown_RESULT']
    '''                        

    mb_cols_to_ordinal=['mb_ZN-smear_STD_RESULT', 'mb_MGIT_STD_RESULT','mb_MGIT_CULTURE_STATUS',
                       'mb_HAIN-test_STD_RESULT', 'mb_MTB-complex_STD_RESULT',
                       'mb_Auramine-smear_STD_RESULT', 'mb_LJ-culture_STD_RESULT','mb_LJ-culture_CULTURE_STATUS',
                       'mb_AccuProbe_STD_RESULT', 'mb_MPT64-Antigen-Test_STD_RESULT',
                       'mb_RT-PCR_STD_RESULT']
                        
    for mb_col in mb_cols_to_ordinal:
        if mb_col in data.columns:
            data.loc[:,mb_col]=data.loc[:,mb_col].replace({'positive':1,'negative':0},regex=True)
    
    ## Calculate log10 values of Colony count and Time to Detection
    #mb_cols_to_log=['mb_Time to Detection_STD_NUM_RESULT','mb_Colony Count_STD_NUM_RESULT']       
    #for mb_col in mb_cols_to_log:
    #    if mb_col in data.columns:
    #        data.loc[:,mb_col]=np.log(data.loc[:,mb_col]+1)

    ## Convert SEX to binary
    if 'SEX' in data.columns:
        data.loc[:,'SEX']=data.loc[:,'SEX'].replace({'M':1,'F':0},regex=True)

    ## Impute missing data + drop datapoints from days without MGIT measurement
    data_for_anal_imp=imputation(data,nan_thr_ratio,only_timepoints_with_mgit)
    print('Imputation done')
    print('mb columns:',data.columns[data.columns.str.startswith('mb')])
    print('data_for_anal_imp.shape after imputation :',data_for_anal_imp.shape)
    del data

    ## Drop columns with all zeroes
    data_for_anal_imp=data_for_anal_imp.loc[:,~data_for_anal_imp.apply(lambda x: all(x==0))]
    print('zeroes dropped')
    print(data_for_anal_imp.shape)

    return data_for_anal_imp,None

    '''
    ## Drop rows with NaNs
    preprocessed_data,rows_with_nans=drop_nans(data_for_anal_imp)
    del data_for_anal_imp

    return preprocessed_data,rows_with_nans
    '''
    
    

## Create the preprocessed data for the patients considered in analysis 
- __use this function for ML method that deal with NaNs__, as there is __no__ interpolation of lab variables + columns and rows with NaNs are __kept__!!

In [14]:
def preprocess_data_without_imputation(filename,pats_for_analysis,common_variables_for_analysis,return_selected_columns_only):
    
    ## Subset whole data to patients consideres in analysis
    data=load_selected_patients_and_vars_for_analysis(filename,pats_for_analysis,\
                                                     common_variables_for_analysis,return_selected_columns_only)  
    #return data 
    
    print('data.shape after loading:',data.shape)
    print(data.loc[:,data.columns.str.startswith('mb_')].columns)

    #print(data[['USUBJID','DAY']])
    ## Drop temporal columns with only 0s and NaNs -> that means column doesn't hold information
    data_temp_cols=data.loc[:,data.columns.str.startswith(('cmind','cmdos','cmday','ae','mh'))]
    temp_cols_to_drop=data_temp_cols.columns[~data_temp_cols.apply(lambda x: x.sum()>0).values]
    data=data.drop(columns=temp_cols_to_drop)

    print('data.shape after dropping temporal cols:',data.shape)

    ## Drop rows with no DAY information 
    data=data[~data['DAY'].isna()]
    print('Data subsetted to patients considered for analysis')   
    print('data.shape after dropping rows with no DAY info:',data.shape)

    ## Replace characters ('[]>') not allowed in column names for XGBoost model
    data.columns=data.columns.str.replace(r'\[REDACTED\]','',regex=False)

    ## Keep only columns with cumulative cmd data (from cmday or cmdos data)
    cm_cols_to_drop=data.columns[data.columns.str.startswith(('cmdos','cmday'))&(data.columns.str.contains('cumul',na=False))]
    data=data.drop(columns=cm_cols_to_drop)

    print('data.shape after keeping only cumul temporal cols :',data.shape)

    '''
    ## Convert mb result columns to ordinal result from positive/negative
    mb_cols_to_ordinal=['mb_Identification_RESULT',',mb_MPT64 Antigen Test_RESULT',
                        'mb_Minimum Cycle Threshold of Detection_RESULT',
                        'mb_Culture Growth_RESULT',
                        'mb_Colony Count, Categorical_RESULT',
                        'mb_Categorical Count_RESULT',
                        'mb_Unknown_RESULT']
                        
    for mb_col in mb_cols_to_ordinal:
        if mb_col in data.columns:
            data.loc[:,mb_col]=data.loc[:,mb_col].replace({'positive':1,'negative':0},regex=True)
    
    ## Calculate log10 values of Colony count and Time to Detection
    mb_cols_to_log=['mb_Time to Detection_STD_NUM_RESULT','mb_Colony Count_STD_NUM_RESULT']       
    for mb_col in mb_cols_to_log:
        if mb_col in data.columns:
            data.loc[:,mb_col]=np.log(data.loc[:,mb_col]+1)

    ## Convert SEX to binary
    if 'SEX' in data.columns:
        data.loc[:,'SEX']=data.loc[:,'SEX'].replace({'M':1,'F':0},regex=True)

    ## Impute missing data + drop datapoints from days without MGIT measurement
    data_for_anal_imp=imputation(data,nan_thr_ratio,only_timepoints_with_mgit)
    print('Imputation done')
    print('data_for_anal_imp.shape after imputation :',data_for_anal_imp.shape)
    del data

    ## Drop columns with all zeroes
    data_for_anal_imp=data_for_anal_imp.loc[:,~data_for_anal_imp.apply(lambda x: all(x==0))]
    print('zeroes dropped')
    print(data_for_anal_imp.shape)

    ## Drop rows with NaNs
    preprocessed_data,rows_with_nans=drop_nans(data_for_anal_imp)
    del data_for_anal_imp
    '''
    return data,None

In [15]:
'''
selection_method='phase'

## Select the common variables and common patients
if selection_method=='phase':
    num_of_common_vars=42
    phase='3'
    with open('../data/common_vars.pickle', 'rb') as f:
        common_vars=pickle.load(f)
    common_variables_for_analysis=common_vars[phase][num_of_common_vars]['common_variables']+['USUBJID','DAY']
    pats_for_analysis=common_vars[phase][num_of_common_vars]['patients']

filename='data_lab_meas_days_only_with_selected_cols.csv.gz'
data,rows_with_nans=preprocess_data(filename,pats_for_analysis,common_variables_for_analysis,\
                     only_timepoints_with_mgit=True,return_selected_columns_only=True)

selection_method='var_clustering'
clust_comb='1-2-4'
num_of_common_vars=34
if 'clustering' in selection_method:
    if selection_method=='patient_clustering':
        fname='pat_clust_common_vars.pickle'
    if selection_method=='var_clustering':
        fname='var_clust_common_vars.pickle'
    
    ## Use patients graphs to extract common patients and variables
    fn=os.path.join('../data/',fname)
    with open(fn, 'rb') as f:
        common_vars=pickle.load(f)
    
    common_variables_for_analysis=common_vars[clust_comb][num_of_common_vars]['common_variables']+['USUBJID','DAY']
    pats_for_analysis=common_vars[clust_comb][num_of_common_vars]['patients']


filename='data_lab_meas_days_only_with_selected_cols.csv.gz'
data,rows_with_nans=preprocess_data(filename,pats_for_analysis,common_variables_for_analysis,\
                     only_timepoints_with_mgit=False,return_selected_columns_only=True)
'''

"\nselection_method='phase'\n\n## Select the common variables and common patients\nif selection_method=='phase':\n    num_of_common_vars=42\n    phase='3'\n    with open('../data/common_vars.pickle', 'rb') as f:\n        common_vars=pickle.load(f)\n    common_variables_for_analysis=common_vars[phase][num_of_common_vars]['common_variables']+['USUBJID','DAY']\n    pats_for_analysis=common_vars[phase][num_of_common_vars]['patients']\n\nfilename='data_lab_meas_days_only_with_selected_cols.csv.gz'\ndata,rows_with_nans=preprocess_data(filename,pats_for_analysis,common_variables_for_analysis,                     only_timepoints_with_mgit=True,return_selected_columns_only=True)\n\nselection_method='var_clustering'\nclust_comb='1-2-4'\nnum_of_common_vars=34\nif 'clustering' in selection_method:\n    if selection_method=='patient_clustering':\n        fname='pat_clust_common_vars.pickle'\n    if selection_method=='var_clustering':\n        fname='var_clust_common_vars.pickle'\n    \n    ## Use p

# EXTRACT RELAPSE FUNCTION

In [1]:
##========================================= 
##========================================= 
### 1. COLLECT PATIENTS, WHO ONLY HAVE FAVOURABLE OUTCOMES AT END OF TREATMENT & AT ALL FOLLOW-UP TIMEPOINTS 
    #. ==>TB-1021: 12 & 18 MONTHS, TB-1022: 18 & 24 MONTHS

### 2. COLLECT PATIENTS, WHO HAVE FAVOURABLE OUTCOME AT END OF TREATMENT, BUT HAVE AT LEAST ONE UNFAVOURABLE OUTCOME AT ANY OF THE FOLLOW-UP TIMEPOINTS 
    #. ==>TB-1021: 12 & 18 MONTHS, TB-1022: 18 & 24 MONTHS

def extract_rifaquin_relapse():

    def load_merged_data_of_lab_vars():
        #load patient IDs who are considered in this  analysis
        pat_id_df=pd.read_csv('../data/patients_in_analysis.csv.gz',index_col=0)
        # get all pat ids
        all_ids=pat_id_df['USUBJID'].to_list()
    
        fname='merged_df.csv.gz'
        
        fn=os.path.join('../data/',fname)
        merged_df=pd.read_csv(fn,low_memory=False,index_col=0)
    
        return merged_df
    
    data=load_merged_data_of_lab_vars()
    arm_df=data.drop_duplicates('USUBJID')[['USUBJID','ARM']].set_index('USUBJID')
    del data
    
    
    de=pd.read_csv('../../C-Path_data/preprocessing/disposition_events.csv',low_memory=True)
    de = de.set_index('USUBJID')


    
    outcome_tb1020 = pd.read_csv('../data/tb_1020_outcome.csv.gz')
    tb_1020_pat_df = outcome_tb1020[outcome_tb1020['UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS'].isin(['FAVOURABLE','RELAPSE'])]
    tb_1020_pat_df = tb_1020_pat_df.rename(columns={'Unnamed: 0':'USUBJID',})

    #df_tb_20 = data[data['USUBJID'].isin(tb_1020_pat_df['USUBJID'].tolist())]



    tb_1020_pat_df = tb_1020_pat_df.set_index('USUBJID')
    tb_1020_pat_df['last_therapy_day'] = de.loc[tb_1020_pat_df.index,'COMPLETION CONTINUATION PHASE'].values
    
    tb_1020_pat_df['RELAPSE']=tb_1020_pat_df['UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS'].replace({'FAVOURABLE':0,'RELAPSE':1})
    tb_1020_pat_df['RELAPSE_DAY']= tb_1020_pat_df['TIME_TO_EVENT'].values
    
    
    tb_1020_pat_df['DAYS_BETWEEEN_THERAPY_END_AND_RELAPSE'] = (tb_1020_pat_df['RELAPSE_DAY'] - tb_1020_pat_df['last_therapy_day']).values
    tb_1020_pat_df.loc[tb_1020_pat_df['RELAPSE']==0,['DAYS_BETWEEEN_THERAPY_END_AND_RELAPSE','RELAPSE_DAY']]=np.nan

    #df_tb_20 = data[data['USUBJID'].isin(tb_1020_pat_df.reset_index()['USUBJID'].tolist())]
   
    tb_1020_pat_df['ARM'] = arm_df.loc[tb_1020_pat_df.reset_index()['USUBJID'],'ARM'].values
    tb_1020_pat_df['STUDYID'] = 'Rifaquin'

    
    return tb_1020_pat_df




def return_fav_unfav_pats_tb_1021(X_subset=None):

    if X_subset is None:
        fn='../data/tb21_22_2984_pats_22_vars_result_at_end_of_treatment_preproc_data_with_imp.csv.gz'
        X_subset=pd.read_csv(fn,index_col=0)
        X_subset=X_subset.rename(columns=lambda x: x.replace('<', 'lower than'))
        X_subset=X_subset.rename(columns=lambda x: x.replace('>', 'higher than'))
    
    de=pd.read_csv('../../C-Path_data/preprocessing/disposition_events.csv',low_memory=True) 
    de=de.set_index('USUBJID')
    
    outcome_tb1021 =pd.read_csv('../data/tb_1021_outcome.csv.gz',index_col=0)
    outcome_tb1021 = outcome_tb1021.loc[X_subset[X_subset['STUDYID']=='TB-1021']['USUBJID'].unique(),:]
    oc_1021 = outcome_tb1021[outcome_tb1021['RESULT_AT_END_OF_TREATMENT']=='FAVOURABLE'].replace('UNASSESSABLE',np.nan)
    
    
    ## TAKE THE LIQUID MEDIUM RESULTS AT MONTH18 AS FINAL RESULTS ==> IF LIQUID MEDIUM IS MISSING, TAKE THE SOLID MEDIUM INSTEAD (CA. 53 PATIENTS)
    oc_1021['RESULT_AT_18_MONTHS'] = oc_1021['RESULT_LIQUID_MEDIUM_AT_18_MONTHS'].values
    oc_1021.loc[oc_1021['RESULT_AT_18_MONTHS'].isna(),'RESULT_AT_18_MONTHS'] = oc_1021.loc[oc_1021['RESULT_AT_18_MONTHS'].isna(),'RESULT_SOLID_MEDIUM_AT_18_MONTHS'].values
    
    
    ## TAKE PATIENTS AS FAVOURABLE WHO HAVE FAVOURABLE LABEL AT ENDO-F-THERAPY + ALL 2 FOLLOW-UP TIMEPOINTS
    out_cols=['RESULT_AT_END_OF_TREATMENT','RESULT_AT_12_MONTHS','RESULT_AT_18_MONTHS']#,'RESULT_AT_24_MONTHS']
    p_1021_fav = (oc_1021.loc[(oc_1021[out_cols]=='FAVOURABLE').all(axis=1),:].index.tolist())

        
    ## ALSO EXTRACT PATIENTS WITHOUT MGIT AT MONTH 18 ==> SOME EARLIER MODELS EXCLUDED THESE PATIENTS
    out_cols=['RESULT_AT_END_OF_TREATMENT','RESULT_AT_12_MONTHS','RESULT_LIQUID_MEDIUM_AT_18_MONTHS']#,'RESULT_AT_24_MONTHS']
    p_1021_fav_ = (oc_1021.loc[(oc_1021[out_cols]=='FAVOURABLE').all(axis=1),:].index.tolist())
    
    pats_wo_mgit_at_18 = (list(set(p_1021_fav) - set(p_1021_fav_)))
    
    ## TAKE PATIENTS AS UNFAVOURABLE WHO HAVE FAVOURABLE LABEL AT END-OF-THERAPY , BUT HAVE AN UNFAVOURABLE OUTCOME AT ANY OF THE 2 FU TIMEPOINTS
    out_cols=['RESULT_AT_12_MONTHS','RESULT_AT_18_MONTHS']#,'RESULT_AT_24_MONTHS']
    p_1021_unfav=oc_1021.loc[(oc_1021['RESULT_AT_END_OF_TREATMENT']=='FAVOURABLE')&\
                             (oc_1021[out_cols]=='UNFAVOURABLE').any(axis=1),:].index.tolist()
    
    
    ## FOR SOME PATIENTS, THE RESULT AT MONTH 12 IS MISSING, BUT THEY HAVE LABELS AT 18 MONTHS 
    ## THESE PATIENTS WERE NOT RETREATED, AND ONLY ONE PATIENTS HAD A DEFAULT 
    p_not_incl=oc_1021.loc[~(oc_1021.index.isin(p_1021_fav+p_1021_unfav))&\
                (oc_1021['RESULT_AT_18_MONTHS']=='FAVOURABLE'),:].index
    
    ## DROP PATIENTS WHO DON'T HAVE COMPLETION DATE OF FOLLOW-UP PHASE
    de_not_incl = de.loc[p_not_incl,:].dropna(how='all',axis=1)
    tb21_no_12_mont_res_but_fav_at_18 = de_not_incl[~de_not_incl['COMPLETION FOLLOW-UP PHASE'].isna()].index.tolist()
    
    ## ADD THESE PATIENTS TO THE FAVOURABLE PATIENT COHORT
    p_1021_fav = p_1021_fav + tb21_no_12_mont_res_but_fav_at_18


    del de

    d={'pats_wo_mgit_at_18':pats_wo_mgit_at_18,
       'tb21_no_12_mont_res_but_fav_at_18':tb21_no_12_mont_res_but_fav_at_18}
    
    return p_1021_fav,p_1021_unfav,d


###===========================
def return_fav_unfav_pats_tb_1022(X_subset):

    if X_subset is None:
        fn='../data/tb21_22_2984_pats_22_vars_result_at_end_of_treatment_preproc_data_with_imp.csv.gz'
        X_subset=pd.read_csv(fn,index_col=0)
        X_subset=X_subset.rename(columns=lambda x: x.replace('<', 'lower than'))
        X_subset=X_subset.rename(columns=lambda x: x.replace('>', 'higher than'))

    outcome_tb1022 = pd.read_csv('../data/tb_1022_outcome.csv.gz',index_col=0)
    outcome_tb1022 = outcome_tb1022.loc[X_subset[X_subset['STUDYID']=='TB-1022']['USUBJID'].unique(),:]
    outcome_tb1022['RESULT_AT_END_OF_TREATMENT']#.value_counts(dropna=False)
    oc_1022 = outcome_tb1022[outcome_tb1022['RESULT_AT_END_OF_TREATMENT']=='FAVOURABLE'].replace('UNASSESSABLE',np.nan)
    
    oc_1022 = oc_1022.replace('NOT ASSESSABLE',np.nan)

    ds=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/ds.csv',low_memory=False)
    ds=ds.loc[ds['USUBJID'].isin(X_subset['USUBJID'].unique())]
    
    de=pd.read_csv('../../C-Path_data/preprocessing/disposition_events.csv',low_memory=True) 
    de=de.set_index('USUBJID')
    
    
    
    p_1022_fav=[]
    out_cols=['RESULT_AT_END_OF_TREATMENT','RESULT_AT_18_MONTHS','RESULT_AT_24_MONTHS','UNFAVOURABLE_OUTCOME_CATEGORY_AT_24_MONTHS']
    p_1022_fav.extend(oc_1022.loc[(oc_1022[out_cols]=='FAVOURABLE').all(axis=1),:].index.tolist())
    
    
    out_cols=['RESULT_AT_18_MONTHS','RESULT_AT_24_MONTHS']#,'UNFAVOURABLE_OUTCOME_CATEGORY_AT_24_MONTHS']
    
    p_1022_unfav=[]
    pats_=oc_1022.loc[(oc_1022['RESULT_AT_END_OF_TREATMENT']=='FAVOURABLE')&\
          (oc_1022[out_cols]=='UNFAVOURABLE').any(axis=1)&\
          #(d['UNFAVOURABLE_OUTCOME_CATEGORY_AT_24_MONTHS']!='FAVOURABLE')&\
          #(~d['UNFAVOURABLE_OUTCOME_CATEGORY_AT_24_MONTHS'].isna())\
          (oc_1022['UNFAVOURABLE_OUTCOME_CATEGORY_AT_24_MONTHS'].isin(['RELAPSE','FAILURE']))\
    
            ,:].index.tolist()
    
    p_1022_unfav.extend(pats_)
    
    
    
    pat_not_incl=oc_1022.loc[~oc_1022.index.isin(p_1022_fav+p_1022_unfav)].index.tolist()
    
    ## RELAPSE
    pats_with_relapse = ds.loc[ds['USUBJID'].isin(pat_not_incl)&\
                               ds['DSTERM'].str.contains('RELAPSE'),:].dropna(how='all',axis=1)
    
    
    pats_with_relapse_with_fav_at_24_month = oc_1022.loc[oc_1022.index.isin(pats_with_relapse['USUBJID'].tolist())&\
                                                        (oc_1022['UNFAVOURABLE_OUTCOME_CATEGORY_AT_24_MONTHS']=='RELAPSE'),:].index.tolist()
    
    relapse_miss_at_18_and_24 = ds.loc[ds['USUBJID'].isin(pat_not_incl)&\
                                       ds['DSTERM'].str.contains('BASED ON CLINICAL AND RADIOLOGIC'),'USUBJID'].tolist()
    
    p_1022_unfav.extend(pats_with_relapse_with_fav_at_24_month)
    p_1022_unfav.extend(relapse_miss_at_18_and_24)
    
    
    
    
    ## DiED
    pats_died = ds.loc[ds['USUBJID'].isin(pat_not_incl)&\
                ds['DSTERM'].str.contains('DEATH|DIED|DECEASED|DCD'),:].dropna(how='all',axis=1)
    
    pats_died_due_to_adverse_event = pats_died.loc[pats_died['DSDECOD'].str.contains('ADVERSE'),['USUBJID','DSSTDY']]
    pats_died_due_to_adverse_event.loc[pats_died_due_to_adverse_event['DSSTDY'].isna(),'DSSTDY']=int(365*16/12)
    pats_died_due_to_other_cause = pats_died.loc[pats_died['DSDECOD'].str.contains('OTHER'),['USUBJID','DSSTDY']]
    
    #pats_died_due_to_adverse_event = pats_died_due_to_adverse_event['USUBJID'].tolist()
    #pats_died_due_to_other_cause = pats_died_due_to_other_cause['USUBJID'].tolist()
    
    
    ## REPROCESSING
    reprocessing = ds.loc[ds['USUBJID'].isin(pat_not_incl)&\
                          ds['DSTERM'].str.contains('REPROCESSING'),['USUBJID','DSSTDY']]
    
    
    ## LOST TO FOLLOW-UP
    str_ = 'CONSENTEMENT RETIRE|LOST|ADVERSE EVENT|CONSENT WITHDRAWN|TRAVEL|WRONGLY EXCLUDED|OTHER'
    pats_lost_to_follow_up = ds.loc[ds['USUBJID'].isin(pat_not_incl)&\
                                    ds['DSTERM'].str.contains(str_),:].dropna(how='all',axis=1)#['USUBJID'].unique().tolist()
    
    
    ## UNKOWN OUTCOME AT 24 MONTHS (SOME OF THEM HAVE UNFAVOURABLE, BUT THE EXACT CATEGORY IS MISSING)
    ## THEY ARE ALL FAVOURABLE AT EOT AND MONTH 18
    unk_outcome_at_24 = ds.loc[ds['USUBJID'].isin(pat_not_incl)&\
                                       ds['DSTERM'].str.contains('COMPLETED'),:].dropna(how='all',axis=1) #'USUBJID'].tolist()
    
    ## RESISTANCE
    pats_with_res = ds.loc[ds['USUBJID'].isin(pat_not_incl)&\
                               ds['DSTERM'].str.contains('MGIT|MDR'),:].dropna(how='all',axis=1)#'USUBJID'].tolist()
    
    
    ## CREATE DICT CONTAINING ALL PATIENTS WHO WERE NOT INCLUDED IN ANALYSIS + REASON WHY
    d={'pats_died_due_to_adverse_event':pats_died_due_to_adverse_event,
       'pats_died_due_to_other_cause':pats_died_due_to_other_cause,
       'pats_lost_to_follow_up':pats_lost_to_follow_up,
       'pats_died_due_to_adverse_event':pats_died_due_to_adverse_event,
      'unk_outcome_at_24':unk_outcome_at_24,
       'pats_with_res':pats_with_res
      }
    
    del de, ds

    return p_1022_fav,p_1022_unfav, d



###====================
def extract_21_22_relapse_pats(include_rifaquin=False,
                              extended_pats=False):

    print('Extracting relapse information...')
    
    ### ====== LOAD ALL PATIENTS DATA =======
    fn='../data/tb21_22_2984_pats_22_vars_result_at_end_of_treatment_preproc_data_with_imp.csv.gz'
    X_subset=pd.read_csv(fn,index_col=0)
    X_subset=X_subset.rename(columns=lambda x: x.replace('<', 'lower than'))
    X_subset=X_subset.rename(columns=lambda x: x.replace('>', 'higher than'))
    
    


    outcome_df=pd.read_csv('../data/tb_1018_20_21_22_30_outcome.csv.gz',index_col=0)
    outcome_df=outcome_df.set_index('USUBJID',drop=True)
    outcome_df=outcome_df.rename(columns={'UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS':'UNFAVOUR_CAT_AT_18_MONTHS'})
    
    df_=outcome_df.reset_index()#
    df_['STUDYID']=df_['USUBJID'].str.split('/',expand=True)[0].values
    df_=df_.set_index('USUBJID')
    df_=df_.loc[X_subset['USUBJID'].unique()]




    if extended_pats==False:

        #### ================================ FAVOURABLE PATIENTS ========== ############
    
        ### COLLECT PATIENTS, WHO ONLY HAVE FAVOURABLE OUTCOMES AT END OF TREATMENT & AT ALL FOLLOW-UP TIMEPOINTS 
        #. ==>TB-1021: 12 & 18 MONTHS, TB-1022: 18 & 24 MONTHS

        pats_with_fav=[]
        
        for study,d in df_[df_['STUDYID'].isin(['TB-1022','TB-1021'])].groupby('STUDYID'):
            if study=='TB-1021':
                out_cols=['RESULT_AT_END_OF_TREATMENT','RESULT_AT_12_MONTHS','RESULT_AT_18_MONTHS']#,'RESULT_AT_24_MONTHS']
                pats_with_fav.extend(d.loc[(d[out_cols]=='FAVOURABLE').all(axis=1),:].index.tolist())
                
            if study=='TB-1022':
                out_cols=['RESULT_AT_END_OF_TREATMENT','RESULT_AT_18_MONTHS','RESULT_AT_24_MONTHS','UNFAVOURABLE_OUTCOME_CATEGORY_AT_24_MONTHS']
                pats_with_fav.extend(d.loc[(d[out_cols]=='FAVOURABLE').all(axis=1),:].index.tolist())

            
        #### ================================ UNFAVOURABLE PATIENTS ========== ############
        
        ### COLLECT PATIENTS, WHO HAVE FAVOURABLE OUTCOME AT END OF TREATMENT, BUT HAVE AT LEAST ONE UNFAVOURABLE OUTCOME AT ANY OF THE FOLLOW-UP TIMEPOINTS 
        #. ==>TB-1021: 12 & 18 MONTHS, TB-1022: 18 & 24 MONTHS
        
        pats_with_unfav=[]
    
        for study,d in df_[df_['STUDYID'].isin(['TB-1022','TB-1021'])].groupby('STUDYID'):
            if study=='TB-1021':
                out_cols=['RESULT_AT_12_MONTHS','RESULT_AT_18_MONTHS']#,'RESULT_AT_24_MONTHS']
        
                pats_=d.loc[(d['RESULT_AT_END_OF_TREATMENT']=='FAVOURABLE')&(d[out_cols]=='UNFAVOURABLE').any(axis=1),:].index.tolist()
                pats_with_unfav.extend(pats_)
                
            if study=='TB-1022':
                out_cols=['RESULT_AT_18_MONTHS','RESULT_AT_24_MONTHS']#,'UNFAVOURABLE_OUTCOME_CATEGORY_AT_24_MONTHS']
                
                pats_=d.loc[(d['RESULT_AT_END_OF_TREATMENT']=='FAVOURABLE')&\
                      (d[out_cols]=='UNFAVOURABLE').any(axis=1)&\
                      (d['UNFAVOURABLE_OUTCOME_CATEGORY_AT_24_MONTHS']!='FAVOURABLE')&\
                      (~d['UNFAVOURABLE_OUTCOME_CATEGORY_AT_24_MONTHS'].isna())\
                        ,:].index.tolist()
                
                pats_with_unfav.extend(pats_)
                
           
    ## EXTENDED PATS: 
    ###. TB-1021:
    ##.  - INCLUDE TB-1021 PATIENTS WHO HAVE MISSING LABEL AT MONTH 12 BUT ARE FAVOURABLE AT MONTH 18 (NO RETREATMENT)
    ##.  - WHERE LIQUID MEDIUM IS MISING AT MONTH 18, USE THE LABEL DERIVED ON SOLID MEDIUM
    ###  TB-1022:
    ##.  - INCLUDE PATIENTS WHO HAVE FAVOURABLE AT MONTH 24 ERRONEOUSLY, BECUASE IN THE DS DATAFRAME, THEY HAVE RELAPSE
    ##.  - EXTRACT PATIENTS WHO HAVE FAVOURABLE EOT OUTCOME, BUT WERE EXCLUDED DUE TO MISSING LABELS DURING FOLLOW-UP
    #.    (I.E. DEATH (DUE TO ADVERSE EVENTS OR OTHER CAUSE), LOST TO FOLLOW-UP, REPROCESSING, REINFECTION, RESISTANCE, UNK. OUTCOME AT MONTH 24 

    if extended_pats==True:
        #pats_with_fav=[]
        p_1021_fav,p_1021_unfav,_ = return_fav_unfav_pats_tb_1021(X_subset)
        p_1022_fav,p_1022_unfav,_ = return_fav_unfav_pats_tb_1022(X_subset)

        pats_with_unfav = p_1021_unfav + p_1022_unfav
        pats_with_fav = p_1021_fav + p_1022_fav
             
    print('pats_with_fav',len(pats_with_fav))
    print('pats_with_unfav',len(pats_with_unfav))
    print(len(pats_with_unfav) + len(pats_with_fav))
    
    
    ## USING THE RAW DISPOSITION EVENTS DATAFRAME (ds) &  PREPROCESSED de DATAFRAME (day of disposition events extracted/patient),
    #. EXTRACT PATIENT IDS:
    #  ==> WHO HAVE DOCUMENTED RELAPSE OR TREATMENT FAILURE (
    #. ==> WHOSE RELAPSE OR TREATMENT FAILURE IS AFTER THE LAST DAY OF THE INITIAL THERAPY
    
    
    #####=========  1. Read the dataframes
    ds=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/ds.csv',low_memory=False)
    ds=ds.loc[ds['USUBJID'].isin(X_subset['USUBJID'].unique())]
    
    de=pd.read_csv('../../C-Path_data/preprocessing/disposition_events.csv',low_memory=True) 
    de=de.set_index('USUBJID')
    
    
    
    #####=========  2. Extract Patients, whose relapse is after the last day available in the initital dataset (==relapse is after initital therapy completion)
    
    # 3 patients didn't have an exact day of relapse==> either 'YES' or 'FOLLOW-UP PHASE' was extracted during preprocessing
    #. => drop these rows temporarily, as they are string values, this way the other RELAPSE days can be converted to float
    de_=de[(~de['RELAPSE'].isin(['YES','FOLLOW-UP PHASE']))&(~de['REINFECTION'].isin(['YES','FOLLOW-UP PHASE']))]
    de_['RELAPSE']=de_['RELAPSE'].astype(float)
    
    ds_rel=(ds.loc[ds['USUBJID'].isin(pats_with_unfav)].groupby('USUBJID').apply(lambda x: x.loc[(x['DSDECOD'].str.contains('RELAPSE'))|((x['DSTERM'].str.contains('RELAPSE')))]))
    #print(ds_rel['DSDECOD'].value_counts(dropna=False))
    
    
    # Extract the last day available in the concatenated clinical & drug regimen dataset ==> this day will be assumed to be the completion day of the initial therapy
    ## N.B.: For TB-1021
    max_days=X_subset.loc[X_subset['USUBJID'].isin(pats_with_unfav),:].groupby('USUBJID').apply(lambda x: x['DAY'].max())
    
    ## Subset de to the unfavourable patient ids
    com_idx=list(set(de_.index)&set(pats_with_unfav))
    de_=de_.loc[(com_idx)]
    
    comm_idx=list(set(de_.index)&set(max_days.index))
    
    ## Add the last day values to the de dataframe
    de_=pd.concat([max_days,de_.loc[comm_idx,:]],axis=1)
    
    ## Add ARMS to the de dataframe
    arms=X_subset.groupby('USUBJID').apply(lambda x:x['ARM'].unique()[0])
    de_['ARM']=arms.loc[de_.index]
    
    ## Column 0 contains tha last days coming from the clinical data
    # => For TB-1021: Last day of therapy (==COMPLETION CONTINUATION PHASE) is available for some patients
    # => Take the maximum of the last day of therapy coming frok clinical data & COMPLETION CONTINUATION PHASE in these cases, as clinical data is sparse
    de_['last_therapy_day']=de_[[0,'COMPLETION CONTINUATION PHASE']].max(axis=1)
    de_.loc[~de_['COMPLETION CONTINUATION PHASE'].isna(),'last_therapy_day']=de_.loc[~de_['COMPLETION CONTINUATION PHASE'].isna(),'COMPLETION CONTINUATION PHASE'].values
    
    
    ## Extract those patients, where relapse was observed after the completion of the initial therapy
    relapse_after_obs_period=de_[(de_['last_therapy_day']<de_['RELAPSE'])]
    
    ## For TB-1021 Study: for the 4 month arms, the continuation phae goes up until 6 months (patients taking placebo after month 4)
    #. => Extract those relapses, that occurred after completion of the 4 month therapy, but before the the end of the 6 month study observation period
    #. => 
    relapse_during_obs_period=de_[(de_['last_therapy_day']>=de_['RELAPSE'])\
                                &(de_['RELAPSE']>100)\
                                &(de_['ARM'].str.contains(r'2MHRZ/2MHR|2EMRZ/2MR'))]
    
    
    #####=========  3. Extracting the IDs of the 3 patients, where the relapse data as either missing, (2 TB-1022 patients), 
    #.                 or it can be imputed from the treatment restart day (TB-1021/2003995)
    idx=de[(de.index.isin(pats_with_unfav))\
        &(de['RELAPSE'].isin(['YES','FOLLOW-UP PHASE']))].index

    
    #print('set(idx)&set(de_.index)',set(idx)&set(de_.index))
    
    pats_with_sparse_relapse_data=pd.DataFrame({'RELAPSE':[238,np.nan,np.nan],
                                                  'ARM':arms.loc[idx],
                                                 'last_therapy_day':de_.loc[idx,'last_therapy_day'].values,
                                                  'STUDYID':de[(de.index.isin(pats_with_unfav))\
                                                                &(de['RELAPSE'].isin(['YES','FOLLOW-UP PHASE']))]['STUDYID'].tolist()},
                                                 index=idx)
    #pats_with_sparse_relapse_data['last_therapy_day']=de_.loc[set(idx)&set(de_.index)],
    
    #####========= 4. Concatenate all relapse patients into 1 dataframe
    #print('relapse_during_obs_period',relapse_during_obs_period)
    pats_with_relapse_df=pd.concat([relapse_after_obs_period[['RELAPSE','STUDYID','ARM','last_therapy_day']],\
                               relapse_during_obs_period[['RELAPSE','STUDYID','ARM','last_therapy_day']],\
                               pats_with_sparse_relapse_data[['RELAPSE','STUDYID','ARM','last_therapy_day']]],axis=0)
    
    pats_with_relapse_df.columns=['RELAPSE_DAY','STUDYID','ARM','last_therapy_day']
    pats_with_relapse_df['RELAPSE']=1


    ## Create dataframe for favourable patients
    max_days_fav=X_subset.loc[X_subset['USUBJID'].isin(pats_with_fav),:].groupby('USUBJID').apply(lambda x: x['DAY'].max())
    arms_fav=X_subset.loc[X_subset['USUBJID'].isin(pats_with_fav),:].groupby('USUBJID').apply(lambda x: x['ARM'].unique()[0])
    study_fav=X_subset.loc[X_subset['USUBJID'].isin(pats_with_fav),:].groupby('USUBJID').apply(lambda x: x['STUDYID'].unique()[0])
    
    pats_wo_relapse_df = pd.DataFrame({'RELAPSE':0},index=pats_with_fav)
    pats_wo_relapse_df['last_therapy_day'] = max_days_fav.loc[pats_wo_relapse_df.index].values
    pats_wo_relapse_df['ARM'] = arms_fav.loc[pats_wo_relapse_df.index].values
    pats_wo_relapse_df['STUDYID'] = study_fav.loc[pats_wo_relapse_df.index].values
    
    pats_relapse_df = pd.concat([pats_with_relapse_df,pats_wo_relapse_df],axis=0)

    pats_relapse_df.index.name='USUBJID'



    
    ## FOR SOME PATIENTS, RETREATMENT DURING FOLLOW-UP STARTED EARLIER AS THE RELAPSE_DAY IN THE DISPOSITION EVENTS
    ## ==> TAKE THE FIRST DAY OF RETREATMENT AS RELAPSE DAYS FOR THESE PATIENTS
    ex = pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/ex.csv', low_memory=False)
    ex = ex[ex['USUBJID'].isin(pats_relapse_df.index.tolist())]
    
    ## Extract patients with retreatmetn during follow-up
    retreatment=ex.loc[ex['EPOCH'].str.contains('FOLLOW',na=False),:].groupby('USUBJID').apply(lambda x: x['EXSTDY'].min()).sort_index().to_frame()
    retreatment_idx=ex.loc[ex['EPOCH'].str.contains('FOLLOW',na=False),'USUBJID'].unique()
    a=pats_relapse_df.loc[retreatment_idx,'RELAPSE_DAY'].sort_index()
    
    ## Concatenate retreatment start day with relapse day coming from disposition events
    b = pd.concat([retreatment,a],axis=1)
    b.columns=['retreatment_start','RELAPSE_DAY']
    
    ## For patients where retreatment started earlier then relapse_day, take the retreatment day as their relapse day
    retreatment_earlier_than_relapse = b[(b['RELAPSE_DAY'] - b['retreatment_start'])>0].index.tolist()
    pats_relapse_df.loc[retreatment_earlier_than_relapse,'RELAPSE_DAY'] = b.loc[retreatment_earlier_than_relapse,'retreatment_start'].values




    ### EXTRACT THE LAST DAY OF THERAPY DRUG ADMINISTRATION USING THE DR_REG DATAFRAME
    #  => FOR SOME REMOXTB PATIENTS, LAST DAY OF THERAPY WAS TAKEN DOWN AS LAST DAY PALCEBO WAS APPLIED
    #  => INSTEAD, EXXTRACT LAST DAY WHERE NON-PLACEBO THERAPY DRUGS WERE APPLIED, TO GET A BETTER SENSE OF RELAPSE AFTER EOT
    month_4_idx = pats_relapse_df[~pats_relapse_df['ARM'].str.contains('Control|2EHRZ')].index.tolist()

    t=pd.read_csv('../data/out_temporal_pat_regimens_1018_20_21_22_30.csv.gz',low_memory=False,index_col=0)
    t = t[t['USUBJID'].isin(month_4_idx)]
    
    max_days = {}


    ## Loop over 4-month patients, and extract the last day of therapy drug adpplication before relapse or if there was no relapse,
    #  take the last day of therapy drug application (250 days are set as a threshold to include patients with extendedn baseline therapy)
    
    for pat_id in (month_4_idx[:]):        

        rel_day = pats_relapse_df.loc[pat_id,'RELAPSE_DAY']#.values

    
        #print('rel_day',rel_day)
    
        ## If no relapse, take the 250 as threshold
        if str(rel_day)=='nan':
            try:
                thr_day = 250 #float(compl_day)
            except ValueError:
                #print(f'VAlueERror: {rel_day} is Nan!')
                continue
    
        ## If relapse, take the relapse day as threshold
        if str(rel_day)!='nan':
            
            try:
                thr_day = np.min([float(rel_day),250])
                
            except ValueError:
                #print(f'VAlueERror: {rel_day} is not Nan!')
                continue
    
        #print('thr_day',thr_day)
        t_ = t[(t['USUBJID']==pat_id) & (t['DAY']<thr_day)]

        ## If there is drug regimen information, extract the maximal number of non-pacebo drugs taken, and then extract the first day where the meximum dose
        #  was reached ==> this was the last day therapy drugs were applied
        if t_.shape[0]>0:
            #print(t_.loc[:,t_.columns.str.contains('num_of_doses')].max().max())
            num_of_doses= t_.loc[:,(t_.columns.str.contains('num_of_doses'))&\
                                   (~t_.columns.str.contains('placebo'))].max().sort_values()
            coln,val = num_of_doses.tail(1).index, num_of_doses.tail(1).values[0]
            last_ther_day = t_.loc[(t_[coln]==val).values,'DAY'].iloc[0]

            max_days[pat_id]=last_ther_day
    
    
    max_days=pd.DataFrame(index=max_days.keys(),data=max_days.values())

    ## For patients
    pats_relapse_df.loc[max_days.index,'last_therapy_day'] = max_days[0].values

    del t



    pats_relapse_df['DAYS_BETWEEEN_THERAPY_END_AND_RELAPSE']=(pats_relapse_df['RELAPSE_DAY'] - pats_relapse_df['last_therapy_day']).values

    if include_rifaquin==True:
        ## ADD RIFAQUIN RELAPSE PATIENTS
        rif_rel = extract_rifaquin_relapse()
        pats_relapse_df = pd.concat([pats_relapse_df,rif_rel[pats_relapse_df.columns]],axis=0)

    return pats_relapse_df#,relapse_during_obs_period,relapse_after_obs_period,pats_with_sparse_relapse_data,max_days